# Phase 2 — results tables & figures in the same format as the Mid-Semester report

Turns the finished runs into tables shaped like **Table 1** and figures styled like **Figure 1**.
No GPU needed: *Runtime → Change runtime type → CPU*. Re-run any time; variants not run yet show `pending`.

Outputs (saved to your Drive folder `quant_study_phase2/report_outputs/`):
* `table1_bnb.md` … `table7_awq.md` — copy-paste-ready tables
* `results_all.csv` — Phase 1 + Phase 2 in the **same columns as `results/results.csv`**
* `fig2_llamacpp_weights.png`, `fig3_kv_cache.png`, `fig4_all_weight_variants.png`

In [ ]:
# ==== 1. Paths ====
import os, json, re
import pandas as pd

USE_SMOKE = False      # True = preview the format with the 10-question smoke-test results

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/quant_study_phase2"
except Exception:
    DRIVE = os.environ.get("QS_DRIVE", "/content/no_drive")   # no Drive: upload phase2_results.zip here
    if os.path.exists("/content/phase2_results.zip"):
        import shutil; shutil.unpack_archive("/content/phase2_results.zip", f"{DRIVE}/runs")
RUN_DIR = f"{DRIVE}/runs_smoke" if USE_SMOKE else f"{DRIVE}/runs"
OUT = f"{DRIVE}/report_outputs"; os.makedirs(OUT, exist_ok=True)
print("reading", RUN_DIR)

In [ ]:
# ==== 2. Phase 1 results (copied from results/results.csv) + Phase 2 results (from the run files) ====
phase1 = pd.DataFrame([
    ("fp16", "gsm8k", 86.5, 19.9, 7.28, 200), ("fp16", "mmlu", 65.5, 18.2, 7.34, 200),
    ("bnb_8bit", "gsm8k", 90.0, 8.9, 4.28, 200), ("bnb_8bit", "mmlu", 65.0, 8.8, 4.36, 200),
    ("bnb_nf4", "gsm8k", 85.0, 15.2, 5.48, 200), ("bnb_nf4", "mmlu", 58.0, 13.6, 5.48, 200),
], columns=["variant", "task", "accuracy_pct", "tokens_per_sec", "peak_mem_gb", "n_questions"])
phase1["runtime"] = "transformers+bnb"

def load_phase2(run_dir):
    rows = []
    if not os.path.isdir(run_dir):
        return pd.DataFrame()
    for fn in sorted(os.listdir(run_dir)):
        m = re.match(r"(.+)__(gsm8k|mmlu)\.jsonl$", fn)
        if not m:
            continue
        v, task = m.groups()
        recs = [json.loads(l) for l in open(f"{run_dir}/{fn}")]
        meta_p = f"{run_dir}/{v}__meta.json"
        meta = json.load(open(meta_p)) if os.path.exists(meta_p) else {}
        rows.append({"variant": v, "task": task,
                     "accuracy_pct": round(100 * sum(r["correct"] for r in recs) / len(recs), 1),
                     "tokens_per_sec": round(meta.get("tokens_per_sec_override") or
                                               sum(r["n_tokens"] for r in recs) / sum(r["wall_s"] for r in recs), 1),
                     "peak_mem_gb": meta.get("peak_mem_gb"), "n_questions": len(recs), "runtime": meta.get("runtime", "llama.cpp"),
                     "kv_buf_gb": meta.get("kv_buf_gb"), "model_buf_gb": meta.get("model_buf_gb")})
    return pd.DataFrame(rows)

phase2 = load_phase2(RUN_DIR)
ALL = pd.concat([phase1, phase2], ignore_index=True)
ALL.to_csv(f"{OUT}/results_all.csv", index=False)
print(ALL.to_string(index=False))
incomplete = ALL[ALL.n_questions < (10 if USE_SMOKE else 200)]
if len(incomplete):
    print("\n!! still incomplete (fewer questions than planned):", sorted(set(incomplete.variant)))

In [ ]:
# ==== 3. Tables in the Table-1 format ====
NAMES = {"fp16": "FP16 (baseline)", "bnb_8bit": "INT8 (bnb)", "bnb_nf4": "NF4 (bnb, 4-bit)",
         "gguf_f16": "FP16 GGUF (baseline)", "gguf_q8_0": "Q8_0 (GGUF)", "gguf_q4_k_m": "Q4_K_M (GGUF, 4-bit)",
         "gguf_f16_kv8": "8-bit KV (q8_0)", "gguf_f16_kv4": "4-bit KV (q4_0)",
         "gguf_mxfp4": "MXFP4 (4-bit)", "gguf_nvfp4": "NVFP4 (4-bit)", "awq_w4": "AWQ (4-bit)",
         "nf4_mlp_only": "NF4 MLP-only", "nf4_attn_only": "NF4 attention-only", "nf4_full": "NF4 full (re-run)"}

def get(v, task, col):
    r = ALL[(ALL.variant == v) & (ALL.task == task)]
    return None if r.empty or pd.isna(r.iloc[0][col]) else float(r.iloc[0][col])

def rng(v, col, unit="", fmt="{:.2f}"):
    # "7.28–7.34 GB" style range across the two tasks, single value if they agree
    vals = [x for x in (get(v, "gsm8k", col), get(v, "mmlu", col)) if x is not None]
    if not vals: return "pending"
    lo, hi = min(vals), max(vals)
    s = fmt.format(lo) if fmt.format(lo) == fmt.format(hi) else f"{fmt.format(lo)}–{fmt.format(hi)}"
    return s + unit

def acc(v, task):
    x = get(v, task, "accuracy_pct"); return "pending" if x is None else f"{x:.1f}%"

def delta(base, new):
    # last column, like "-1.5 pp" / "-25%" / "~25–30% slower"
    out = {}
    for task in ("gsm8k", "mmlu"):
        a, b = get(base, task, "accuracy_pct"), get(new, task, "accuracy_pct")
        out[task] = "—" if None in (a, b) else f"{b - a:+.1f} pp"
    def pct(col):
        ps = []
        for task in ("gsm8k", "mmlu"):
            a, b = get(base, task, col), get(new, task, col)
            if None not in (a, b): ps.append(100 * (b - a) / a)
        return ps
    m = pct("peak_mem_gb")
    out["mem"] = "—" if not m else f"{sum(m)/len(m):+.0f}%"
    t = pct("tokens_per_sec")
    if not t: out["tps"] = "—"
    else:
        lo, hi = sorted([abs(round(x)) for x in t])[0], sorted([abs(round(x)) for x in t])[-1]
        word = "faster" if sum(t) > 0 else "slower"
        out["tps"] = f"~{lo}% {word}" if lo == hi else f"~{lo}–{hi}% {word}"
    return out

def make_table(title, variants, base, compare, delta_label, extra_rows=()):
    d = delta(base, compare)
    header = ["Metric"] + [NAMES[v] for v in variants] + [delta_label]
    rows = [["GSM8K accuracy"] + [acc(v, "gsm8k") for v in variants] + [d["gsm8k"]],
            ["MMLU accuracy"] + [acc(v, "mmlu") for v in variants] + [d["mmlu"]],
            ["Peak GPU memory"] + [rng(v, "peak_mem_gb", " GB") for v in variants] + [d["mem"]],
            ["Throughput (tok/s)"] + [rng(v, "tokens_per_sec", "", "{:.1f}") for v in variants] + [d["tps"]]]
    rows += [list(r) for r in extra_rows]
    df = pd.DataFrame(rows, columns=header)
    mdtxt = f"**{title}**\n\n| " + " | ".join(header) + " |\n|" + "---|" * len(header) + "\n"
    mdtxt += "\n".join("| " + " | ".join(r) + " |" for r in rows) + "\n"
    return df, mdtxt

def kv_row():
    vals = [rng(v, "kv_buf_gb", " GB") if v.startswith("gguf") else "—" for v in ["gguf_f16", "gguf_f16_kv8", "gguf_f16_kv4"]]
    a, b = get("gguf_f16", "gsm8k", "kv_buf_gb"), get("gguf_f16_kv4", "gsm8k", "kv_buf_gb")
    return ["KV-cache size (4k ctx)"] + vals + ["—" if None in (a, b) else f"{100*(b-a)/a:+.0f}%"]

TABLES = {
  "table1_bnb": ("Table 1: Accuracy, memory and throughput by precision (transformers + bitsandbytes, Mid-Sem)",
                 ["fp16", "bnb_8bit", "bnb_nf4"], "fp16", "bnb_nf4", "4-bit vs FP16", ()),
  "table3_gguf": ("Table 3: GGUF k-quants under llama.cpp",
                 ["gguf_f16", "gguf_q8_0", "gguf_q4_k_m"], "gguf_f16", "gguf_q4_k_m", "4-bit vs FP16", ()),
  "table4_kv_cache": ("Table 4: KV-cache quantization (FP16 weights, llama.cpp)",
                 ["gguf_f16", "gguf_f16_kv8", "gguf_f16_kv4"], "gguf_f16", "gguf_f16_kv4", "4-bit KV vs FP16 KV", "KV"),
  "table5_fp4": ("Table 5: FP4 microscaling formats (llama.cpp, emulated on T4)",
                 ["gguf_f16", "gguf_mxfp4", "gguf_nvfp4"], "gguf_f16", "gguf_nvfp4", "NVFP4 vs FP16", ()),
  "table7_awq": ("Table 7: AWQ vs NF4 — two 4-bit methods, same library (transformers)",
                 ["fp16", "bnb_nf4", "awq_w4"], "fp16", "awq_w4", "AWQ vs FP16", ()),
  "table8_ablation": ("Table 8: Ablation — which half of the network causes the NF4 accuracy loss?",
                 ["fp16", "nf4_attn_only", "nf4_mlp_only", "bnb_nf4"], "fp16", "nf4_mlp_only", "MLP-only vs FP16", ()),
  "table6_cross_runtime": ("Table 6: Same bit-width, different library (bitsandbytes vs llama.cpp)",
                 ["bnb_8bit", "gguf_q8_0", "bnb_nf4", "gguf_q4_k_m"], "bnb_nf4", "gguf_q4_k_m", "Q4_K_M vs NF4", ()),
}
md_all = ""
for key, (title, vs, base, cmp, dl, extra) in TABLES.items():
    df, mdtxt = make_table(title, vs, base, cmp, dl, [kv_row()] if extra == "KV" else ())
    open(f"{OUT}/{key}.md", "w").write(mdtxt); md_all += mdtxt + "\n"
    print(title); print(df.to_string(index=False)); print()
open(f"{OUT}/all_tables.md", "w").write(md_all)

In [ ]:
# ==== 4. Figures in the Figure-1 style (same colours, markers, 3 panels) ====
import matplotlib.pyplot as plt

def plot(variants, fname, title):
    sub_all = ALL[ALL.variant.isin(variants)].copy()
    if sub_all.empty:
        print("nothing to plot yet for", fname); return
    order = [v for v in variants if v in set(sub_all.variant)]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for task, marker, color in [("gsm8k", "o", "#2563eb"), ("mmlu", "s", "#dc2626")]:
        sub = sub_all[sub_all.task == task].set_index("variant").reindex(order)
        for ax, col in zip(axes, ["accuracy_pct", "peak_mem_gb", "tokens_per_sec"]):
            ax.plot(order, sub[col], marker=marker, label=task, color=color, linewidth=2, markersize=8)
    axes[0].set_title("Accuracy (%) by Variant", fontweight="bold"); axes[0].set_ylabel("Accuracy %")
    axes[1].set_title("Peak GPU Memory (GB) by Variant", fontweight="bold"); axes[1].set_ylabel("GB")
    axes[2].set_title("Throughput (tokens/sec) by Variant", fontweight="bold"); axes[2].set_ylabel("tokens/sec")
    for ax in axes:
        ax.tick_params(axis="x", rotation=20); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(title, fontweight="bold"); plt.tight_layout()
    plt.savefig(f"{OUT}/{fname}", dpi=150); plt.show()

plot(["gguf_f16", "gguf_q8_0", "gguf_q4_k_m", "gguf_mxfp4", "gguf_nvfp4"], "fig2_llamacpp_weights.png",
     "Phi-4-mini-instruct: llama.cpp Weight Quantization Trade-off (T4 GPU)")
plot(["gguf_f16", "gguf_f16_kv8", "gguf_f16_kv4"], "fig3_kv_cache.png",
     "Phi-4-mini-instruct: KV-cache Quantization Trade-off (llama.cpp, T4 GPU)")
plot(["fp16", "bnb_8bit", "bnb_nf4", "awq_w4", "gguf_f16", "gguf_q8_0", "gguf_q4_k_m", "gguf_mxfp4", "gguf_nvfp4"],
     "fig4_all_weight_variants.png", "Phi-4-mini-instruct: All Weight-Quantization Variants (T4 GPU)")

In [ ]:
# ==== 5. Download everything as one zip ====
import shutil
shutil.make_archive("/content/report_outputs", "zip", OUT)
print("saved:", OUT, "and /content/report_outputs.zip (Files panel on the left -> right-click -> Download)")